# Preprocessing

This notebook contains the full cleaning pipeline inline so it can run without importing helper code from elsewhere in the project.

In [ ]:
import os, sys, logging, random
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional

import numpy as np
import pandas as pd


def get_logger(name: str = "ecommerce_llm", level: int = logging.INFO) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(level)
    if not logger.handlers:
        handler = logging.StreamHandler(sys.stdout)
        formatter = logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
            datefmt="%H:%M:%S",
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)
    return logger


logger = get_logger("notebook03")


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


@dataclass
class Config:
    project_root: Path = field(
        default_factory=lambda: Path(os.environ.get("PROJECT_ROOT", Path.cwd().parent)).resolve()
    )
    seed: int = 42
    data_dir_name: str = "data"
    raw_subdir: str = "raw"
    processed_subdir: str = "processed"
    sample_subdir: str = "sample"

    def __post_init__(self) -> None:
        set_seed(self.seed)

    @property
    def data_dir(self) -> Path:
        return self.project_root / self.data_dir_name

    @property
    def raw_data_dir(self) -> Path:
        return self.data_dir / self.raw_subdir

    @property
    def processed_data_dir(self) -> Path:
        return self.data_dir / self.processed_subdir

    @property
    def sample_data_dir(self) -> Path:
        return self.data_dir / self.sample_subdir

    def ensure_dirs(self) -> None:
        for path in (self.raw_data_dir, self.processed_data_dir, self.sample_data_dir):
            path.mkdir(parents=True, exist_ok=True)


_HTML_TAG_RE = __import__("re").compile(r"<[^>]+>")
_URL_RE = __import__("re").compile(r"https?://\S+|www\.\S+")
_MULTI_SPACE_RE = __import__("re").compile(r"\s+")
_NOISE_RE = __import__("re").compile(r"[^\w\s.,!?'\"%$€£@#:/()-]")


def remove_html(text: str) -> str:
    return _HTML_TAG_RE.sub(" ", text)


def remove_urls(text: str) -> str:
    return _URL_RE.sub(" ", text)


def normalize_unicode(text: str) -> str:
    import unicodedata
    text = unicodedata.normalize("NFKC", text)
    return "".join(ch for ch in text if ch.isprintable())


def normalize_whitespace(text: str) -> str:
    return _MULTI_SPACE_RE.sub(" ", text).strip()


def remove_noise_chars(text: str, keep_punct: bool = True) -> str:
    if keep_punct:
        return _NOISE_RE.sub(" ", text)
    return __import__("re").sub(r"[^\w\s]", " ", text)


def clean_text(text: str, lowercase: bool = False) -> str:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""
    text = str(text)
    text = remove_html(text)
    text = remove_urls(text)
    text = normalize_unicode(text)
    text = remove_noise_chars(text)
    text = normalize_whitespace(text)
    if lowercase:
        text = text.lower()
    return text


def word_count(text: str) -> int:
    return len(text.split()) if isinstance(text, str) else 0


def save_jsonl(records, path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(__import__("json").dumps(rec, ensure_ascii=False) + "\n")
    logger.info(f"Saved {sum(1 for _ in open(path, encoding='utf-8'))} records -> {path}")


INSTRUCTION_ALIASES = ["instruction", "question", "query", "customer", "input", "text"]
RESPONSE_ALIASES = ["response", "answer", "response_text", "output", "reply"]


class Preprocessor:
    def __init__(self, min_words: int = 2, max_words: int = 512):
        self.min_words = min_words
        self.max_words = max_words

    @staticmethod
    def _resolve_column(df: pd.DataFrame, aliases: List[str]) -> Optional[str]:
        for alias in aliases:
            if alias in df.columns:
                return alias
        return None

    def standardize_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        instr_col = self._resolve_column(df, INSTRUCTION_ALIASES)
        resp_col = self._resolve_column(df, RESPONSE_ALIASES)
        if instr_col is None or resp_col is None:
            raise ValueError(f"Could not resolve instruction/response columns from {list(df.columns)}")
        out = df.rename(columns={instr_col: "instruction", resp_col: "response"})
        keep_cols = [c for c in ["instruction", "response", "category"] if c in out.columns]
        return out[keep_cols].copy()

    def remove_duplicates(self, df: pd.DataFrame) -> pd.DataFrame:
        before = len(df)
        df = df.drop_duplicates(subset=["instruction", "response"]).reset_index(drop=True)
        logger.info(f"Removed {before - len(df)} duplicate rows ({before} -> {len(df)}).")
        return df

    def clean_columns(self, df: pd.DataFrame, lowercase: bool = False) -> pd.DataFrame:
        df = df.copy()
        df["instruction"] = df["instruction"].apply(lambda t: clean_text(t, lowercase=lowercase))
        df["response"] = df["response"].apply(lambda t: clean_text(t, lowercase=lowercase))
        return df

    def remove_empty_rows(self, df: pd.DataFrame) -> pd.DataFrame:
        before = len(df)
        df = df[(df["instruction"].str.len() > 0) & (df["response"].str.len() > 0)].reset_index(drop=True)
        logger.info(f"Removed {before - len(df)} empty rows ({before} -> {len(df)}).")
        return df

    def filter_by_length(self, df: pd.DataFrame) -> pd.DataFrame:
        before = len(df)

        def ok(row):
            iw, rw = word_count(row["instruction"]), word_count(row["response"])
            return self.min_words <= iw <= self.max_words and self.min_words <= rw <= self.max_words

        df = df[df.apply(ok, axis=1)].reset_index(drop=True)
        logger.info(f"Removed {before - len(df)} rows outside length bounds ({before} -> {len(df)}).")
        return df

    def run(self, df: pd.DataFrame, lowercase: bool = False) -> pd.DataFrame:
        logger.info(f"Starting preprocessing on {len(df)} raw rows.")
        df = self.standardize_columns(df)
        df = self.remove_duplicates(df)
        df = self.clean_columns(df, lowercase=lowercase)
        df = self.remove_empty_rows(df)
        df = self.filter_by_length(df)
        df = self.remove_duplicates(df)
        logger.info(f"Preprocessing complete: {len(df)} clean rows remain.")
        return df

    def build_instruction_response_pairs(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.run(df)


## 2. Run the full cleaning pipeline

This notebook contains the full cleaning pipeline inline so it can run without importing helper code from elsewhere in the project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

cfg = Config()

raw_path = cfg.raw_data_dir / "ecommerce_support_raw.csv"
df_raw = pd.read_csv(raw_path)
print(f"Loaded {len(df_raw):,} raw rows")
df_raw.head(3)


Loaded 26,872 raw rows


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...


## 2. Run the full cleaning pipeline

Steps (see `src/preprocess.py::Preprocessor.run`):
1. Standardize columns → `instruction`, `response`
2. Remove exact duplicates
3. Clean text (HTML, URLs, unicode, noisy chars, whitespace)
4. Remove empty rows post-clean
5. Filter rows outside min/max word bounds
6. Remove duplicates again (post-clean collisions)

In [ ]:
preprocessor = Preprocessor(min_words=2, max_words=400)
df_clean = preprocessor.run(df_raw, lowercase=False)

print(f"\nFinal clean dataset: {len(df_clean):,} rows (from {len(df_raw):,} raw rows)")
df_clean.head(5)


13:00:15 | INFO     | src.preprocess | Starting preprocessing on 26872 raw rows.
13:00:15 | INFO     | src.preprocess | Removed 0 duplicate rows (26872 -> 26872).
13:00:18 | INFO     | src.preprocess | Removed 0 empty rows (26872 -> 26872).
13:00:18 | INFO     | src.preprocess | Removed 5 rows outside length bounds (26872 -> 26867).
13:00:18 | INFO     | src.preprocess | Removed 0 duplicate rows (26867 -> 26867).
13:00:18 | INFO     | src.preprocess | Preprocessing complete: 26867 clean rows remain.

Final clean dataset: 26,867 rows (from 26,872 raw rows)


,instruction,response,category
0,question about cancelling order Order Number,I've understood you have a question regarding ...,ORDER
1,i have a question about cancelling oorder Orde...,I've been informed that you have a question ab...,ORDER
2,i need help cancelling puchase Order Number,I can sense that you're seeking assistance wit...,ORDER
3,I need to cancel purchase Order Number,I understood that you need assistance with can...,ORDER
4,"I cannot afford this order, cancel purchase Or...",I'm sensitive to the fact that you're facing f...,ORDER


## 3. Sanity checks

In [37]:
assert df_clean["instruction"].str.len().min() > 0, "Found empty instruction after cleaning!"
assert df_clean["response"].str.len().min() > 0, "Found empty response after cleaning!"
assert df_clean.duplicated(subset=["instruction", "response"]).sum() == 0, "Duplicates remain!"

print("All sanity checks passed ✅")
print("\nExample cleaned pair:")
print("INSTRUCTION:", df_clean.iloc[0]["instruction"])
print("RESPONSE:   ", df_clean.iloc[0]["response"])


All sanity checks passed ✅

Example cleaned pair:
INSTRUCTION: question about cancelling order Order Number
RESPONSE:    I've understood you have a question regarding canceling order Order Number , and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.


## 4. Save cleaned instruction-response pairs

In [ ]:
out_path = cfg.processed_data_dir / "cleaned_pairs.jsonl"
save_jsonl(df_clean.to_dict(orient="records"), out_path)

# Also keep a small human-readable CSV sample for quick spot-checks
sample_path = cfg.sample_data_dir / "cleaned_sample.csv"
df_clean.sample(min(50, len(df_clean)), random_state=cfg.seed).to_csv(sample_path, index=False)
print(f"Saved sample preview -> {sample_path}")


13:02:52 | INFO     | src.utils | Saved 26867 records -> C:\Users\Asus\Downloads\Ecommerce-LLM-Finetuning\Ecommerce-LLM-Finetuning\data\processed\cleaned_pairs.jsonl
Saved sample preview -> C:\Users\Asus\Downloads\Ecommerce-LLM-Finetuning\Ecommerce-LLM-Finetuning\data\sample\cleaned_sample.csv


## Summary

- Cleaned raw text (HTML/URL/unicode/noise/whitespace)
- Removed duplicate and empty rows
- Filtered by length bounds
- Saved clean instruction-response pairs to `data/processed/cleaned_pairs.jsonl`

Next: `04_prompt_formatting.ipynb` to convert these pairs into the training prompt format and tokenize them.